# grant_finder
## Notebook 1: Data Acquisition & Exploration

The full pipeline spans five notebooks:
- **Notebook 1 (this notebook):** Data acquisition and exploration
- **Notebook 2:** Data cleaning and feature engineering
- **Notebook 3:** Embedding and vector search
- **Notebook 4:** Matching pipeline and RAG
- **Notebook 5:** Evaluation and demo

### This Notebook
This notebook covers the acquisition and initial exploration of FY2023 federal financial assistance data from [USASpending.gov](https://www.usaspending.gov).


---
## 1. API Exploration

I first explored the USASpending REST API (`/api/v2/search/spending_by_award/`) to understand what data was available before committing to a bulk download approach.

In [4]:
import requests
import pandas as pd
import json

url = 'https://api.usaspending.gov/api/v2/search/spending_by_award/'

payload = {
    'filters': {
        'award_type_codes': ['04'],
        'time_period': [{'start_date': '2023-01-01', 'end_date': '2024-12-31'}],
        'recipient_type_names': ['nonprofit']  # correct field - recipient_type_filters does NOT work
    },
    'fields': [
        'Recipient Name', 'recipient_id', 'Award Amount',
        'Awarding Agency', 'CFDA Number', 'CFDA Title',
        'Award Type', 'Start Date', 'Place of Performance State Code'
    ],
    'page': 1,
    'limit': 100,
    'sort': 'Award Amount',
    'order': 'desc'
}

response = requests.post(url, json=payload)
df_api = pd.DataFrame(response.json()['results'])
df_api.head(10)

,internal_id,Recipient Name,recipient_id,Award Amount,Awarding Agency,CFDA Number,CFDA Title,Award Type,Start Date,Place of Performance State Code,awarding_agency_id,agency_slug,generated_internal_id
0,254636879,CLIMATE UNITED FUND,22ead7ad-8b8b-23f5-52f5-6ac5a09e92fd-R,6.970000e+09,Environmental Protection Agency,66.957,None,PROJECT GRANT (B),2024-04-01,MD,700,environmental-protection-agency,ASST_NON_84094001_068
1,254636940,COALITION FOR GREEN CAPITAL,0266f5d3-9617-6908-0f49-d5a20a3fef55-R,5.000000e+09,Environmental Protection Agency,66.957,None,PROJECT GRANT (B),2024-04-01,DC,700,environmental-protection-agency,ASST_NON_84094201_068
2,254636848,OPPORTUNITY FINANCE NETWORK,a9c16abd-b31e-b44c-adf4-643444d4d529-C,2.290000e+09,Environmental Protection Agency,66.960,None,PROJECT GRANT (B),2024-04-01,PA,700,environmental-protection-agency,ASST_NON_84093901_068
3,254637480,"POWER FORWARD COMMUNITIES, INC",77b00ab8-a37a-e838-76bf-1b6005b6cbce-R,2.000000e+09,Environmental Protection Agency,66.957,None,PROJECT GRANT (B),2024-04-01,MD,700,environmental-protection-agency,ASST_NON_84096001_068
4,254636970,"INCLUSIV, INC",fd0ec47a-24cb-3fad-0467-fec154be9921-C,1.870000e+09,Environmental Protection Agency,66.960,None,PROJECT GRANT (B),2024-04-01,NY,700,environmental-protection-agency,ASST_NON_84094301_068
5,254636909,"JUSTICE CLIMATE FUND, INC.",7c8c4ab7-57a8-ba33-df7c-074944205c8a-R,9.400000e+08,Environmental Protection Agency,66.960,None,PROJECT GRANT (B),2024-04-01,DC,700,environmental-protection-agency,ASST_NON_84094101_068
6,200059053,LOS ANGELES COUNTY OFFICE OF EDUCATION,2ca7f9f3-35a8-b5ed-aea1-ac07942a45ee-C,8.399273e+08,Department of Health and Human Services,93.600,None,PROJECT GRANT (B),2019-07-01,CA,806,department-of-health-and-human-services,ASST_NON_09CH011157_075
7,267881780,THE MENTAL HEALTH ASSOCIATION OF NEW YORK CITY...,7f43999d-c0f7-8d6a-5413-78804c5662a6-C,7.859847e+08,Department of Health and Human Services,93.243,None,PROJECT GRANT (B),2021-09-30,NY,806,department-of-health-and-human-services,ASST_NON_H79SM084816_075
8,260200661,HOOSIER ENERGY RURAL ELECTRIC COOPERATIVE INC,3771bd1f-72a2-a498-01ab-b993560bc709-C,6.767964e+08,Department of Agriculture,10.758,None,PROJECT GRANT (B),2024-09-30,IN,95,department-of-agriculture,ASST_NON_CLSS00000088093_012
9,260200752,"WOLVERINE POWER SUPPLY COOPERATIVE, INC.",9374b9eb-d22b-a2bb-b867-8fe2be7384b7-R,6.516240e+08,Department of Agriculture,10.758,None,PROJECT GRANT (B),2024-09-30,MI,95,department-of-agriculture,ASST_NON_CLSS00000088194_012


### API Findings

After exploration, I identified several limitations that made the API unsuitable as the primary data source:

1. **`recipient_type_filters` is not a valid parameter** — the correct field is `recipient_type_names`, but even with the correct field, filtering was unreliable and returned state government agencies instead of nonprofits.

2. **`CFDA Title` is always null via the API** — this field is critical for describing what each grant program funds and is needed for the embedding layer. It is not returned by the search endpoint.

3. **`transaction_description` is not available via the API** — this is the richest text field and the primary input for semantic matching. The API does not expose it.

4. **Results are biased toward large awards** — sorting by amount returns $100M+ block grants to state agencies, not the competitive project grants relevant to nonprofits and small businesses.

5. **No total record count** — pagination uses a cursor pattern with no way to know how many total records exist, making bulk extraction impractical.

**Decision:** Use the USASpending bulk download instead. The full FY2023 financial assistance archive provides all fields including `cfda_title` and `transaction_description`, is properly filterable locally, and covers every award rather than a biased sample.

---
## 2. Bulk Data Download

I downloaded the full FY2023 financial assistance archive from the [USASpending Award Data Archive](https://www.usaspending.gov/download_center/award_data_archive). This file covers all federal financial assistance awards for fiscal year 2023.

In [5]:
# Download the FY2023 full financial assistance archive
!wget -O assistance_2023.zip "https://files.usaspending.gov/award_data_archive/FY2023_All_Assistance_Full_20260406.zip"

# Unzip and check file sizes
!unzip assistance_2023.zip
!ls -lh

--2026-05-08 23:29:48--  https://files.usaspending.gov/award_data_archive/FY2023_All_Assistance_Full_20260406.zip
Resolving files.usaspending.gov (files.usaspending.gov)... 166.123.8.118, 2610:108:4100:100c::9:168
Connecting to files.usaspending.gov (files.usaspending.gov)|166.123.8.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1259114111 (1.2G) [binary/octet-stream]
Saving to: ‘assistance_2023.zip’

assistance_2023.zip 100%[===================>]   1.17G  35.2MB/s    in 82s     

2026-05-08 23:31:11 (14.6 MB/s) - ‘assistance_2023.zip’ saved [1259114111/1259114111]

Archive:  assistance_2023.zip
  inflating: FY2023_All_Assistance_Full_20260407_1.csv  
  inflating: FY2023_All_Assistance_Full_20260407_2.csv  
  inflating: FY2023_All_Assistance_Full_20260407_3.csv  
  inflating: FY2023_All_Assistance_Full_20260407_4.csv  
  inflating: FY2023_All_Assistance_Full_20260407_5.csv  
  inflating: FY2023_All_Assistance_Full_20260407_6.csv  
  inflating: FY2023_All_

The archive unpacks into 7 CSV files totaling ~11GB uncompressed. This is too large to load into memory at once, so I use chunked processing to filter down to only the records I need.

---
## 3. Schema Exploration

Before filtering, I examine the schema to identify the relevant columns.

In [6]:
# Read a small sample to inspect schema without loading the full file
df_sample = pd.read_csv('FY2023_All_Assistance_Full_20260407_1.csv', nrows=10000, low_memory=False)

df_sample.columns.tolist()

['assistance_transaction_unique_key',
 'assistance_award_unique_key',
 'award_id_fain',
 'modification_number',
 'award_id_uri',
 'sai_number',
 'federal_action_obligation',
 'total_obligated_amount',
 'total_outlayed_amount_for_overall_award',
 'indirect_cost_federal_share_amount',
 'non_federal_funding_amount',
 'total_non_federal_funding_amount',
 'face_value_of_loan',
 'original_loan_subsidy_cost',
 'total_face_value_of_loan',
 'total_loan_subsidy_cost',
 'generated_pragmatic_obligations',
 'disaster_emergency_fund_codes_for_overall_award',
 'outlayed_amount_from_COVID-19_supplementals_for_overall_award',
 'obligated_amount_from_COVID-19_supplementals_for_overall_award',
 'outlayed_amount_from_IIJA_supplemental_for_overall_award',
 'obligated_amount_from_IIJA_supplemental_for_overall_award',
 'action_date',
 'action_date_fiscal_year',
 'period_of_performance_start_date',
 'period_of_performance_current_end_date',
 'awarding_agency_code',
 'awarding_agency_name',
 'awarding_sub_agen

In [7]:
df_sample['business_types_description'].value_counts().head(10)

,count
business_types_description,
INDIVIDUAL,8301
FOR-PROFIT ORGANIZATION (OTHER THAN SMALL BUSINESS),448
SMALL BUSINESS,301
OTHER,267
INDEPENDENT SCHOOL DISTRICT,144
NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION),72
STATE GOVERNMENT,64
PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER EDUCATION,61
PRIVATE INSTITUTION OF HIGHER EDUCATION,36


In [8]:
df_sample['assistance_type_description'].value_counts()

,count
assistance_type_description,
"DIRECT PAYMENT WITH UNRESTRICTED USE (RETIREMENT, PENSION, VETERANS BENEFITS, ETC.) (D)",7376
"DIRECT PAYMENT FOR SPECIFIED USE, AS A SUBSIDY OR OTHER NON-REIMBURSABLE DIRECT FINANCIAL AID (C)",1699
"OTHER REIMBURSABLE, CONTINGENT, INTANGIBLE, OR INDIRECT FINANCIAL ASSISTANCE",452
GUARANTEED/INSURED LOAN (F),395
PROJECT GRANT (B),40
COOPERATIVE AGREEMENT (B),20
FORMULA GRANT (A),12
DIRECT LOAN (E),4
BLOCK GRANT (A),2


In [9]:
# Null rates for key text fields
df_sample[['cfda_title', 'transaction_description', 'funding_opportunity_goals_text']].isna().sum()

,0
cfda_title,0
transaction_description,0
funding_opportunity_goals_text,9983


In [10]:
df_sample['transaction_description'].dropna().iloc[0]

'THE PURPOSE OF THIS AGREEMENT, BETWEEN THE U.S. DEPARTMENT OF AGRICULTURE, CENTER FOR FAITH BASED AND NEIGHBORHOOD PARTNERSHIPS AND RAFI-USA IS TO BUILD THE CAPACITY OF FAITH AND COMMUNITY-BASED ORGANIZATIONS FROM ACROSS THE COUNTRY BY PROVIDING LOCAL LEADERS THE RESOURCES AND TECHNICAL ASSISTANCE TO ASSIST IN ACHIEVING THEIR GOALS AND OBJECTIVES.'

### Schema Findings

The dataset has 112 columns. The fields relevant to this project are:

| Field | Purpose | Null Rate |
|---|---|---|
| `recipient_uei` | Unique org identifier | Low |
| `recipient_name` | Org name | Low |
| `recipient_state_code` | Org location | Low |
| `business_types_description` | Org category (nonprofit, university, etc.) | Low |
| `cfda_number` | Grant program identifier | Low |
| `cfda_title` | Grant program name | **Zero** |
| `transaction_description` | Free text describing funded work | **Zero** |
| `assistance_type_description` | Award type (project grant, block grant, etc.) | Low |
| `federal_action_obligation` | Award amount | Low |
| `awarding_agency_name` | Federal agency | Low |

Notably, `funding_opportunity_goals_text` is almost entirely null or contains "NOT APPLICABLE" and is dropped. The `transaction_description` field is the primary text signal with zero nulls, averaging ~1,800 characters per record.

---
## 4. Filtering & Chunked Processing

I applied two filters:

1. **Organization type:** Nonprofits, institutions of higher education, and small businesses — the three categories most likely to actively seek grants using a tool like this.

2. **Award type:** Project Grants and Cooperative Agreements only — competitive discretionary awards. This excludes block grants (pass-through to states), direct payments, loans, and other non-competitive assistance types.

Because the files total ~11GB, I process them in 50,000-row chunks and load only the 15 columns needed.

In [11]:
import gc

files = [
    'FY2023_All_Assistance_Full_20260407_1.csv',
    'FY2023_All_Assistance_Full_20260407_2.csv',
    'FY2023_All_Assistance_Full_20260407_3.csv',
    'FY2023_All_Assistance_Full_20260407_4.csv',
    'FY2023_All_Assistance_Full_20260407_5.csv',
    'FY2023_All_Assistance_Full_20260407_6.csv',
    'FY2023_All_Assistance_Full_20260407_7.csv',
]

cols_to_keep = [
    'recipient_uei',
    'recipient_name',
    'recipient_state_code',
    'recipient_city_name',
    'recipient_zip_code',
    'awarding_agency_name',
    'cfda_number',
    'cfda_title',
    'assistance_type_description',
    'transaction_description',
    'federal_action_obligation',
    'period_of_performance_start_date',
    'period_of_performance_current_end_date',
    'business_types_description',
    'primary_place_of_performance_state_name',
]

# Filter 1: org type
org_filter = 'NONPROFIT|INSTITUTION OF HIGHER EDUCATION|SMALL BUSINESS'

# Filter 2: competitive grant types only
grant_filter = 'PROJECT GRANT|COOPERATIVE AGREEMENT'

all_results = []

for f in files:
    print(f'Processing {f}...')
    file_count = 0

    for chunk in pd.read_csv(f, chunksize=50000, usecols=cols_to_keep, low_memory=False):
        filtered = chunk[
            chunk['business_types_description'].str.contains(org_filter, case=False, na=False) &
            chunk['assistance_type_description'].str.contains(grant_filter, case=False, na=False)
        ]
        if len(filtered) > 0:
            all_results.append(filtered)
            file_count += len(filtered)

    print(f'  -> Found {file_count:,} rows')
    gc.collect()

df = pd.concat(all_results, ignore_index=True)
print(f'\nTotal rows: {len(df):,}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

df.to_csv('grant_recipients_2023.csv', index=False)
print('Saved to grant_recipients_2023.csv')

Processing FY2023_All_Assistance_Full_20260407_1.csv...
  -> Found 47,791 rows
Processing FY2023_All_Assistance_Full_20260407_2.csv...
  -> Found 84,119 rows
Processing FY2023_All_Assistance_Full_20260407_3.csv...
  -> Found 55,294 rows
Processing FY2023_All_Assistance_Full_20260407_4.csv...
  -> Found 49,502 rows
Processing FY2023_All_Assistance_Full_20260407_5.csv...
  -> Found 29,397 rows
Processing FY2023_All_Assistance_Full_20260407_6.csv...
  -> Found 8,029 rows
Processing FY2023_All_Assistance_Full_20260407_7.csv...
  -> Found 1,465 rows

Total rows: 275,597
Memory usage: 909.8 MB
Saved to grant_recipients_2023.csv


---
## 5. Dataset Exploration

In [12]:
df = pd.read_csv('grant_recipients_2023.csv')
df.shape

(275597, 15)

In [13]:
df['business_types_description'].value_counts().head(10)

,count
business_types_description,
PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER EDUCATION,100093
NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION),84510
PRIVATE INSTITUTION OF HIGHER EDUCATION,57879
SMALL BUSINESS,9811
FOR-PROFIT ORGANIZATION (OTHER THAN SMALL BUSINESS),4557
NONPROFIT WITHOUT 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION),3448
NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION);OTHER,2720
PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER EDUCATION;STATE GOVERNMENT,2610
PRIVATE INSTITUTION OF HIGHER EDUCATION;NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION);OTHER,2449


In [14]:
df['cfda_title'].value_counts().head(20)

,count
cfda_title,
BIOMEDICAL RESEARCH AND RESEARCH TRAINING,11774
ALLERGY AND INFECTIOUS DISEASES RESEARCH,10704
CARDIOVASCULAR DISEASES RESEARCH,10471
AGING RESEARCH,9306
EXTRAMURAL RESEARCH PROGRAMS IN THE NEUROSCIENCES AND NEUROLOGICAL DISORDERS,7963
"DIABETES, DIGESTIVE, AND KIDNEY DISEASES EXTRAMURAL RESEARCH",7378
HEAD START,6698
MENTAL HEALTH RESEARCH GRANTS,6431
BASIC AND APPLIED SCIENTIFIC RESEARCH,5612


In [15]:
df['awarding_agency_name'].value_counts().head(10)

,count
awarding_agency_name,
Department of Health and Human Services,152187
National Science Foundation,28550
Department of Defense,17019
Department of the Interior,11343
Department of Energy,9514
Department of Agriculture,9380
Department of Education,8285
National Aeronautics and Space Administration,6605
Department of Commerce,4272


In [16]:
df['federal_action_obligation'].describe()

,federal_action_obligation
count,2.755970e+05
mean,4.036676e+05
std,8.378670e+06
min,-2.000000e+09
25%,0.000000e+00
50%,5.500000e+04
75%,3.712360e+05
max,3.143000e+09


In [17]:
df[['recipient_uei', 'recipient_name', 'cfda_number']].nunique()

,0
recipient_uei,27399
recipient_name,26904
cfda_number,1325


In [18]:
grants_per_org = df.groupby('recipient_uei')['cfda_number'].count()
programs_per_org = df.groupby('recipient_uei')['cfda_number'].nunique()

grants_per_org.describe()

,cfda_number
count,27399.000000
mean,10.058615
std,92.285186
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,4104.000000


In [19]:
programs_per_org.value_counts().head(10)

,count
cfda_number,
1,19995
2,3748
3,1460
4,636
5,335
6,228
7,135
8,109
9,71


In [20]:
# How many orgs have meaningful grant history?
pd.Series({
    'Orgs with 3+ grants': (grants_per_org >= 3).sum(),
    'Orgs with 5+ grants': (grants_per_org >= 5).sum()
})

,0
Orgs with 3+ grants,8907
Orgs with 5+ grants,4635


In [21]:
repeat_grants = df.groupby(['recipient_uei', 'cfda_number']).size().reset_index(name='count')
repeat_grants['count'].value_counts().head(5)

,count
count,
1,33494
2,12114
3,5633
4,3083
5,1839


In [22]:
# Transaction description quality
df['desc_length'] = df['transaction_description'].str.len()
print('--- Transaction Description Length ---')
print(df['desc_length'].describe())

print('\n--- Sample Records ---')
for i in range(3):
    print(f'\n[{df["recipient_name"].iloc[i]}]')
    print(f'Type: {df["business_types_description"].iloc[i]}')
    print(f'CFDA: {df["cfda_number"].iloc[i]} - {df["cfda_title"].iloc[i]}')
    print(f'Agency: {df["awarding_agency_name"].iloc[i]}')
    print(f'Amount: ${df["federal_action_obligation"].iloc[i]:,.0f}')
    print(f'Description: {df["transaction_description"].iloc[i][:300]}')

--- Transaction Description Length ---
count    275590.000000
mean       1812.240589
std        1396.252388
min           1.000000
25%         119.000000
50%        2209.000000
75%        3074.000000
max       10052.000000
Name: desc_length, dtype: float64

--- Sample Records ---

[RURAL ADVANCEMENT FOUNDATION INTERNATIONAL - USA]
Type: NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION)
CFDA: 10.889 - BUILDING THE CAPACITY OF FAITH AND COMMUNITY-BASED ORGANIZATIONS
Agency: Department of Agriculture
Amount: $500,000
Description: THE PURPOSE OF THIS AGREEMENT, BETWEEN THE U.S. DEPARTMENT OF AGRICULTURE, CENTER FOR FAITH BASED AND NEIGHBORHOOD PARTNERSHIPS AND RAFI-USA IS TO BUILD THE CAPACITY OF FAITH AND COMMUNITY-BASED ORGANIZATIONS FROM ACROSS THE COUNTRY BY PROVIDING LOCAL LEADERS THE RESOURCES AND TECHNICAL ASSISTANCE T

[THE TRUST FOR TOMORROW]
Type: NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION)
CFDA: 10.931 - AGRICULTU

In [23]:
# CFDA program statistics - do programs fund multiple orgs?
# This validates the recommender approach: if programs are recurring and broad,
# we can learn patterns from historical recipients
cfda_stats = df.groupby(['cfda_number', 'cfda_title']).agg(
    num_recipients=('recipient_uei', 'nunique'),
    total_awarded=('federal_action_obligation', 'sum'),
    avg_award=('federal_action_obligation', 'mean')
).reset_index().sort_values('num_recipients', ascending=False)

print('Top CFDA programs by number of unique recipients:')
print(cfda_stats.head(20).to_string())

Top CFDA programs by number of unique recipients:
      cfda_number                                                                                 cfda_title  num_recipients  total_awarded     avg_award
752        45.024                              PROMOTION OF THE ARTS GRANTS TO ORGANIZATIONS AND INDIVIDUALS            2562   8.239912e+07  2.676165e+04
1051       93.243  SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PROJECTS OF REGIONAL AND NATIONAL SIGNIFICANCE            1626   1.294676e+09  2.621332e+05
314        14.191                                                   MULTIFAMILY HOUSING SERVICE COORDINATORS            1529   1.013619e+08  3.188484e+04
1178       93.600                                                                                 HEAD START            1122   8.121238e+09  1.212487e+06
772        47.084                                               NSF TECHNOLOGY, INNOVATION, AND PARTNERSHIPS            1035   6.447670e+08  3.415079e+05
867        81.049         

---
## 6. Data Cleaning

I remove award records that are modifications or amendments to existing grants rather than new awards. These are identified by a zero or negative `federal_action_obligation`.

In [24]:
df_clean = df[df['federal_action_obligation'] > 0].copy()

pd.Series({
    'before': len(df),
    'after': len(df_clean),
    'removed': len(df) - len(df_clean)
})

,0
before,275597
after,169132
removed,106465


In [25]:
df_clean['desc_length'] = df_clean['transaction_description'].str.len()
df_clean['desc_length'].describe()

,desc_length
count,169129.000000
mean,1883.066151
std,1383.649125
min,1.000000
25%,134.000000
50%,2313.000000
75%,3098.000000
max,10052.000000


In [26]:
# Sample short descriptions
df_clean[df_clean['desc_length'] < 100][['transaction_description', 'cfda_title', 'desc_length']].sample(5, random_state=42)

,transaction_description,cfda_title,desc_length
102022,EDUCATIONAL OPPORTUNITY CENTER THAT WILL SERVE...,TRIO EDUCATIONAL OPPORTUNITY CENTERS,81.0
29948,HEAD START AND EARLY HEAD START,HEAD START,31.0
21523,MULTIFAMILY HOUSING SERVICE COORDINATORS,MULTIFAMILY HOUSING SERVICE COORDINATORS,40.0
65851,UNIVERSITY OF DELAWARE - FY23 CONGRESSIONAL CO...,CONGRESSIONAL GRANTS,69.0
204028,LOCOMOTION AND TRANSITIONS OF AN AMPHIBIOUS SY...,BASIC AND APPLIED SCIENTIFIC RESEARCH,71.0


---
## 7. Building Org Profiles

I aggregate the award-level data to create one profile per organization. The key field is `all_descriptions` — a concatenation of every `transaction_description` for that org across all their FY2023 awards. This becomes the semantic fingerprint of what the org does, used for embedding in Notebook 3.

I also retain structured fields (org type, state, funding history, CFDA programs received) for the structural similarity layer.

In [27]:
org_profiles = df_clean.groupby('recipient_uei').agg(
    org_name=('recipient_name', 'first'),
    org_type=('business_types_description', 'first'),
    state=('recipient_state_code', 'first'),
    city=('recipient_city_name', 'first'),
    num_grants=('cfda_number', 'count'),
    num_programs=('cfda_number', 'nunique'),
    total_funding=('federal_action_obligation', 'sum'),
    avg_award=('federal_action_obligation', 'mean'),
    agencies=('awarding_agency_name', lambda x: list(x.unique())),
    cfda_programs=('cfda_number', lambda x: list(x.unique())),
    cfda_titles=('cfda_title', lambda x: list(x.unique())),
    all_descriptions=('transaction_description', lambda x: ' '.join(x.dropna()))
).reset_index()

org_profiles['desc_length'] = org_profiles['all_descriptions'].str.len()

print(f'Org profiles built: {len(org_profiles):,}')

main_types = [
    'NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION)',
    'PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER EDUCATION',
    'PRIVATE INSTITUTION OF HIGHER EDUCATION',
    'SMALL BUSINESS'
]
subset = org_profiles[org_profiles['org_type'].isin(main_types)]
print(f'\n--- Description Length by Main Org Type ---')
print(subset.groupby('org_type')['desc_length'].describe().to_string())
print(f'\nOrgs with short combined descriptions (<200 chars): {(org_profiles["desc_length"] < 200).sum():,} ({(org_profiles["desc_length"] < 200).sum()/len(org_profiles)*100:.1f}%)')

Org profiles built: 23,111

--- Description Length by Main Org Type ---
                                                                                   count          mean            std   min     25%     50%       75%        max
org_type                                                                                                                                                        
NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION)  14816.0   5625.443237   69619.048624   4.0   81.00   214.0   2235.25  4262689.0
PRIVATE INSTITUTION OF HIGHER EDUCATION                                           1058.0  52141.957467  331849.510662  10.0  125.25   406.0   4937.25  5600283.0
PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER EDUCATION                           1247.0  94313.270249  437436.121601   5.0  177.00  2597.0  10444.50  5801474.0
SMALL BUSINESS                                                                    3534.0   2614.614035    3016.092944   0.0

In [28]:
# Sample a well-documented org profile
sample = org_profiles[org_profiles['num_programs'] >= 5].iloc[0]

sample[['org_name', 'org_type', 'state', 'num_grants', 'num_programs', 'total_funding', 'desc_length']]

,13
org_name,UNIVERSITY OF NORTH CAROLINA AT GREENSBORO
org_type,PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER ...
state,NC
num_grants,72
num_programs,42
total_funding,26112515.62
desc_length,134205


In [29]:
sample['cfda_titles']

['CHILD CARE ACCESS MEANS PARENTS IN SCHOOL',
 'TRANS-NIH RESEARCH SUPPORT',
 'MENTAL AND BEHAVIORAL HEALTH EDUCATION AND TRAINING GRANTS',
 'OFFICE OF SCIENCE FINANCIAL ASSISTANCE PROGRAM',
 'AGING RESEARCH',
 'STEM EDUCATION (FORMERLY EDUCATION AND HUMAN RESOURCES)',
 'MATHEMATICAL AND PHYSICAL SCIENCES',
 'ENGLISH LANGUAGE ACQUISITION STATE GRANTS',
 'CHILD HEALTH AND HUMAN DEVELOPMENT EXTRAMURAL RESEARCH',
 'NURSE ANESTHETIST TRAINEESHIP',
 'COMPREHENSIVE CENTERS',
 'TEACHER QUALITY PARTNERSHIP GRANTS',
 'TRIO MCNAIR POST-BACCALAUREATE ACHIEVEMENT',
 'BIOMEDICAL RESEARCH AND RESEARCH TRAINING',
 'BIOLOGICAL SCIENCES',
 'ENGINEERING',
 'ALCOHOL RESEARCH PROGRAMS',
 'EDUCATION RESEARCH, DEVELOPMENT AND DISSEMINATION',
 'NURSE FACULTY LOAN PROGRAM (NFLP)',
 'GEOSCIENCES',
 'ADVANCED NURSING EDUCATION WORKFORCE GRANT PROGRAM',
 'NURSE EDUCATION, PRACTICE QUALITY AND RETENTION GRANTS',
 'BASIC SCIENTIFIC RESEARCH',
 'DISCOVERY AND APPLIED RESEARCH FOR TECHNOLOGICAL INNOVATIONS TO IMPROV

In [30]:
sample['all_descriptions'][:500]

'ACCESS TO CHILD CARE ENHANCES STUDENT SUCCESS (ACCESS) PROJECT COMPARATIVE BIOACTIVITY MODELING AND BIOINFORMATICS TO STUDY COMBINATION EFFECTS IN COMPLEX NATURAL PRODUCT MIXTURES - PROJECT SUMMARY/ABSTRACT PLANTS HAVE BEEN USED TO TREAT ILLNESSES SINCE ANTIQUITY, WITH THE FIRST DOCUMENTED CASE OF THIS PRACTICE DATING BACK MORE THAN 4,000 YEARS. BOTANICAL MEDICINES ARE STILL A MAJOR FORM OF HEALTHCARE AROUND THE WORLD, AND IN THE US ARE WIDELY USED IN THE FORM OF HERBAL DIETARY SUPPLEMENTS. THE '

In [31]:
# Saving org profiles
org_profiles.to_csv('org_profiles.csv', index=False)
print('Saved to org_profiles.csv')
print(f'Shape: {org_profiles.shape}')

Saved to org_profiles.csv
Shape: (23111, 14)


In [32]:
from google.colab import drive
drive.mount('/content/drive')

df.to_csv('/content/drive/MyDrive/grant_finder/grant_recipients_2023.csv', index=False)
org_profiles.to_csv('/content/drive/MyDrive/grant_finder/org_profiles.csv', index=False)

Mounted at /content/drive
